# 汽车排序问题

**类别:** 调度

来源: [https://www.hexaly.com/templates/car-sequencing](https://www.hexaly.com/templates/car-sequencing)


## 问题描述

**汽车排序问题** 涉及安排一组汽车的生产顺序。这些汽车并非完全相同,在基本车型的基础上有不同的配置选项可供选择。装配线上有不同的工作站来安装各种选项(空调、天窗等)。这些工作站有最大产能限制,它们最多能处理装配线上一定比例的车辆。因此,必须将汽车排成一个序列,以使每个工作站的产能都不会被超过。例如,如果某个工作站最多能处理装配线上三分之二的车辆,那么在序列中任意连续 3 辆车的窗口内,至多只能有 2 辆车需要该选项。

	

### 学到的要点

- 使用 [列表决策变量](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html) 来表示汽车的序列
- 区分 [结构性约束与第一优先级目标](https://www.hexaly.com/docs/last/modelingprinciples/modelingprinciples.html#distinguish-constraints-from-first-priority-objectives)
- 使用 [非线性算子](https://www.hexaly.com/docs/last/mathematicaloperators/operatorsreference.html#table-of-available-operators-and-functions) 来计算违规次数


## 数据

我们提供的汽车排序问题实例来自 [CSPLib](http://www.csplib.org/Problems/prob001/)。数据文件的格式如下:

- 第 1 行: 车辆数量、选项数量、类别数量
- 第 2 行: 对于每个选项,一个块中包含该选项的最大车辆数
- 第 3 行: 对于每个选项,该最大车辆数所对应的块大小
- 然后,对于每个类别:

- 类别的索引
- 该类别中的车辆数量
- 对于每个选项,该类别是否需要它(1 或 0)。


## 模型

汽车排序问题的 Hexaly 模型使用一个 [列表决策变量](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html) 来表示汽车的序列。列表的第 i 个元素对应于第 i 辆要生产的汽车的索引。使用 [**partition**](https://www.hexaly.com/docs/last/mathematicaloperators/collectionvariables.html#n-ary-operators) 算子,可以确保待生产的每辆汽车都出现在装配线上。

根据这个序列,我们可以对每个选项和装配线上的每个位置,计算出从该位置开始的窗口中具有该选项的车辆数量。然后我们可以推导出每个选项和每个窗口的违规数量。

尽管这个问题是一个纯可行性问题,我们仍然选择添加一个目标,即使所有选项和所有窗口的产能违规次数之和最小化。事实上,无产能违规更像是“业务”约束,而不是结构性约束。如果出现少量违规,装配线会减速但仍可继续运行。相反,在装配线上的某个位置同时放置两辆车在物理上是不可能的:这是一个结构性约束。有关高优先级目标和硬约束之间区别的更多信息,请参阅文档中的 [此章节](https://www.hexaly.com/docs/last/modelingprinciples/modelingprinciples.html#distinguish-constraints-from-first-priority-objectives)。


## Python 实现


In [ ]:
from pathlib import Path

from optagent import ModelBuilder, solve


def read_instance(filename):
    data = [int(value) for value in Path(filename).read_text().split()]
    iterator = iter(data)
    nb_positions, nb_options, nb_classes = next(iterator), next(iterator), next(iterator)
    max_per_window = [next(iterator) for _ in range(nb_options)]
    window_size = [next(iterator) for _ in range(nb_options)]
    options, initial = [], []
    for class_id in range(nb_classes):
        next(iterator)
        count = next(iterator)
        options.append([next(iterator) for _ in range(nb_options)])
        initial.extend([class_id] * count)
    return nb_positions, max_per_window, window_size, options, initial


def main(instance_file, time_limit=20):
    nb_positions, max_per_window, window_size, options_data, initial = read_instance(instance_file)
    model = ModelBuilder()
    sequence = model.list(nb_positions, name="sequence")
    model.constraint(sequence.count() == nb_positions)
    classes, options = model.array(initial), model.array(options_data)
    violations = []
    for option, (limit, width) in enumerate(zip(max_per_window, window_size)):
        for start in range(nb_positions - width + 1):
            count = model.sum(
                *(options[classes[sequence.at(position)]][option] for position in range(start, start + width))
            )
            violations.append(model.max(count - limit, 0))
    total_violations = model.sum(*violations)
    model.minimize(total_violations, name="total_violations")
    solution = solve(model, time_limit_s=float(time_limit))
    values = solution.values({"violations": total_violations, "sequence": sequence})
    print(f"Violations = {values['violations']}; Status = {solution.status.value}")
    print(" ".join(str(initial[index]) for index in values["sequence"]))
    return solution

## 运行实例


In [ ]:
INSTANCE_DIR = Path.cwd() / "instances"

In [ ]:
solution = main(INSTANCE_DIR / "4_72.in", time_limit=5)